# Sprint 5 – Notebook 13 – Preventing Data Leakage


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
rng = np.random.default_rng(42)
df = pd.DataFrame({'Age': rng.integers(18, 65, 120), 'Income': rng.integers(25000, 120000, 120)})
df['Purchased'] = ((df['Age'] > 35) & (df['Income'] > 60000)).astype(int)
X = df[['Age', 'Income']]
y = df['Purchased']
print(df.head())

   Age  Income  Purchased
0   22   66763          0
1   54   88498          1
2   48   87170          1
3   38   69754          1
4   38  106232          1


## 1. What is Data Leakage?

### Definition
Data leakage happens when a model learns information it should not have at prediction time. It makes model evaluation look better than real performance. Leakage can be direct or subtle. Preventing it is essential for trustworthy ML.

### Business Example
A shop wants to predict whether a customer will buy tomorrow. The model accidentally uses tomorrow’s receipt total. This reveals the outcome too early. The reported score cannot be trusted.

### AI/ML Use
Define when a prediction is made. Keep future and target-derived information out of features. Fit preprocessing using training data only. Evaluate on a realistic holdout set.

### Python Example

In [2]:
leaky_df = df.copy()
leaky_df["Future_Receipt"] = leaky_df["Purchased"]
print("Target revealed:", (leaky_df["Future_Receipt"] == leaky_df["Purchased"]).all())

Target revealed: True


### Explanation
The added column exactly reveals the target. This would make prediction artificially easy. In production, the value is unavailable before purchase. It must not be used as a feature.

## 2. Target Leakage

### Definition
Target leakage occurs when an input contains the answer or information derived from it. It may be an exact target copy or a later outcome. Such features can dominate model learning. Removing them is necessary for realistic predictions.

### Business Example
A loan default model includes the final collection status. This status is known only after default events occur. It reveals the label. The model cannot use it when a loan is first approved.

### AI/ML Use
Audit features for target-derived information. Ask when each value becomes available. Exclude post-outcome columns. Compare performance before and after removing suspicious features.

### Python Example

In [5]:
target_copy = y.copy()
print("Correlation with target:", target_copy.corr(y))

Correlation with target: 1.0


### Explanation
The target copy has perfect correlation with the label. This is a warning sign, not proof that every perfect correlation is leakage. Feature timing and meaning must be checked. Here the copied label is clearly invalid.

## 3. Train-Test Contamination

### Definition
Train-test contamination happens when test information enters model training. This can occur through shared rows, duplicates, or preprocessing. It makes the test set less independent. Scores may become overly optimistic.

### Business Example
A customer appears in both training and test data with nearly identical records. The model may memorize the customer. Test accuracy then looks stronger than performance on new customers. Group-aware splitting may be needed.

### AI/ML Use
Remove exact duplicates carefully before splitting when appropriate. Use group splits for repeated entities. Keep test data out of feature engineering decisions. Fit transformations only on training folds.

### Python Example

In [7]:
train_ids = {101, 102, 103}
test_ids = {103, 104}
print("Overlapping customer IDs:", train_ids.intersection(test_ids))

Overlapping customer IDs: {103}


### Explanation
Customer 103 appears in both sets. This overlap may contaminate evaluation. The split should account for customer identity. Not all repeated IDs are duplicates; group-aware partitioning is often better.

## 4. Feature Leakage

### Definition
Feature leakage means an input feature contains information unavailable when predictions are required. It may involve future actions or operational outcomes. The feature can look highly predictive during training. In production it may not exist yet.

### Business Example
A churn model uses a customer’s account-closure date. This date is only recorded after the customer leaves. It predicts churn perfectly but too late. The feature is unusable for early warning.

### AI/ML Use
Build a feature availability table. Confirm timestamps and data collection processes. Remove features recorded after the prediction moment. Validate the full inference workflow.

### Python Example

In [9]:
example = pd.DataFrame({"Churned": [0, 1, 0], "Closure_Recorded": [0, 1, 0]})
print(example.corr(numeric_only=True))

                  Churned  Closure_Recorded
Churned               1.0               1.0
Closure_Recorded      1.0               1.0


### Explanation
The closure flag is perfectly correlated with churn. Its meaning shows why it is suspicious. Correlation alone does not establish leakage. The decisive issue is when the feature becomes available.

## 5. Preprocessing Leakage

### Definition
Preprocessing leakage occurs when transformation parameters are learned from validation or test data. Examples include scaling, imputation, and feature selection before splitting. Even transformations without target labels can leak distribution information. Pipelines help enforce correct order.

### Business Example
A company fills missing salaries using the mean of all customers. The test customers affect that mean. The training process has indirectly seen test information. The correct mean comes only from training customers.

### AI/ML Use
Split before fitting any imputer, scaler, encoder, or feature selector. Fit on training only. Transform held-out data with the same fitted object. Repeat fitting within each cross-validation fold.

### Python Example

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
wrong_mean = X["Income"].mean()
right_mean = X_train["Income"].mean()
print("Whole-data mean (wrong):", round(wrong_mean))
print("Training-only mean (correct):", round(right_mean))

Whole-data mean (wrong): 72550
Training-only mean (correct): 74132


### Explanation
The two means may differ because the test set contains different values. Using the whole-data mean allows test information into preprocessing. Training-only statistics avoid this contamination. The fitted imputer or scaler should be reused unchanged on test rows.

## 6. Temporal Leakage

### Definition
Temporal leakage occurs when future information is used to predict earlier events. It is common in forecasting and time-based customer models. Random splitting can mix future and past observations. Chronological evaluation is often necessary.

### Business Example
A shop forecasts weekly demand using next week’s actual sales. This makes the forecast appear accurate. Those sales are not known when the forecast is created. The model must use only past observations.

### AI/ML Use
Use time-aware features and chronological splits. Avoid centered rolling averages that include future rows. Use lagged features based on past values. Respect data publication delays.

### Python Example

In [13]:
ts = pd.DataFrame({"Sales": [10, 12, 15, 11, 18]})
ts["Past_Sales"] = ts["Sales"].shift(1)
print(ts)

   Sales  Past_Sales
0     10         NaN
1     12        10.0
2     15        12.0
3     11        15.0
4     18        11.0


### Explanation
The shifted column contains the previous observation. It does not look ahead to future sales. The first row is missing because no earlier value exists. This is a simple safe lag feature.

## 7. Examples of Data Leakage

### Definition
Leakage can appear as a copied target, future record, duplicated customer, or global preprocessing statistic. Different examples require different fixes. High scores alone do not prove leakage. Understanding the prediction workflow is essential.

### Business Example
A fraud model uses chargeback results recorded weeks later. A churn model uses an account-closure date. A demand model uses future sales totals. All three reveal information too late.

### AI/ML Use
Review each feature’s source and timestamp. Audit duplicated or grouped records. Check whether validation influenced model fitting. Test a simpler leakage-free baseline.

### Python Example

In [15]:
examples = {"Target copy": "Remove", "Future sales": "Use past lags",
            "Whole-data scaler": "Fit on train only"}
for issue, fix in examples.items():
    print(issue, "→", fix)

Target copy → Remove
Future sales → Use past lags
Whole-data scaler → Fit on train only


### Explanation
The examples show three common leakage mechanisms. Each requires a different correction. Some columns should be removed. Others can be replaced with time-safe features or training-only transformations.

## 8. How to Detect Leakage

### Definition
Leakage detection combines technical checks with domain knowledge. Unexpectedly perfect scores are a warning sign. Features recorded after the target event deserve scrutiny. Duplicates and preprocessing order should also be reviewed.

### Business Example
An analyst gets almost 100% fraud detection accuracy. They inspect the feature list and find a later investigation outcome. That feature reveals the label. Removing it produces a more realistic score.

### AI/ML Use
Check suspiciously high correlations and feature importance. Inspect feature timestamps and availability. Search for duplicate entities across splits. Review every fit operation and cross-validation pipeline.

### Python Example

In [17]:
suspect = df.assign(Target_Copy=df["Purchased"])
print(suspect.corr(numeric_only=True)["Purchased"].sort_values(ascending=False))

Purchased      1.000000
Target_Copy    1.000000
Income         0.628358
Age            0.509410
Name: Purchased, dtype: float64


### Explanation
The target copy has correlation 1 with the target. This is a useful alert for further investigation. Some valid features can also correlate strongly. The final decision depends on meaning and timing.

## 9. How to Prevent Leakage

### Definition
Prevent leakage by defining the prediction moment and splitting correctly. Fit all learned preprocessing steps on training data only. Keep target-derived and future features out of inputs. Use a final untouched test set for honest evaluation.

### Business Example
A company predicts customer churn at the start of each month. It uses only information available before that date. It splits customers or time periods appropriately. The deployed pipeline repeats the same transformations.

### AI/ML Use
Use sklearn Pipeline and ColumnTransformer where appropriate. Perform feature selection inside cross-validation. Use group or temporal splits when needed. Document which features exist at inference time.

### Python Example

In [19]:
safe = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
safe.fit(X_train, y_train)
print("Held-out accuracy:", round(accuracy_score(y_test, safe.predict(X_test)), 3))

Held-out accuracy: 0.867


### Explanation
The pipeline fits the scaler on training rows as part of model fitting. The test data is only transformed and predicted. This avoids preprocessing leakage. The score is more meaningful than one obtained with a contaminated workflow.